# XGBoost Weather Classification

Goal: Compare XGBoost with the existing models using the same
airport dataset, 16 input features, labels, and chronological splits.

Train: 2015–2021
Validation: 2022–2023
Test: 2024–2025

The test set stays reserved until model selection is complete.

Current results are preliminary because shared preprocessing
includes future-looking cleaning that needs to be corrected.

In [ ]:
%pip install xgboost pandas scikit-learn

In [ ]:
from pathlib import Path

import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

candidates = [
    Path("../data/processed/unv_model_ready.csv"),
    Path("data/processed/unv_model_ready.csv"),
]

DATA_PATH = next(
    (path for path in candidates if path.exists()),
    None,
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Cannot find unv_model_ready.csv. "
        "Check that notebook 02 has generated it."
    )

df = pd.read_csv(
    DATA_PATH,
    index_col="valid",
    parse_dates=True,
).sort_index()

CLASS_NAMES = ["Clear", "Changing", "Precipitation"]

FEATURES = [
    "temp_c", "rh_pct", "dew_spread_c",
    "dp_1h", "dp_3h", "dp_6h",
    "dtemp_3h", "drh_3h",
    "dp_12h", "dp_24h", "dp_accel", "dspread_3h",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
]

assert df.index.is_unique, "Duplicate timestamps found."
assert not df[FEATURES + ["label"]].isna().any().any()
assert set(df["label"].unique()) == {0, 1, 2}

print("Loaded:", DATA_PATH.resolve())
print("Dataset shape:", df.shape)
print("Number of inputs:", len(FEATURES))

df.head()

In [ ]:
BUFFER = pd.Timedelta(hours=12)
TRAIN_END = pd.Timestamp("2022-01-01")
VAL_END = pd.Timestamp("2024-01-01")

train = df.loc[df.index < TRAIN_END - BUFFER].copy()

val = df.loc[
    (df.index >= TRAIN_END)
    & (df.index < VAL_END - BUFFER)
].copy()

test = df.loc[df.index >= VAL_END].copy()

for name, part in [
    ("Train", train),
    ("Validation", val),
    ("Test", test),
]:
    print(
        f"{name}: {len(part):,} rows | "
        f"{part.index.min()} to {part.index.max()}"
    )

X_train = train[FEATURES]
y_train = train["label"].astype(int)

X_val = val[FEATURES]
y_val = val["label"].astype(int)

# Leave test inputs and labels unused during development.

In [ ]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=500,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=4,
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

In [ ]:
val_pred = xgb_model.predict(X_val)

print("Validation accuracy:",
      round(accuracy_score(y_val, val_pred), 3))

print("Validation macro-F1:",
      round(f1_score(y_val, val_pred, average="macro"), 3))

print(
    classification_report(
        y_val,
        val_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0,
    )
)

pd.DataFrame(
    confusion_matrix(y_val, val_pred, labels=[0, 1, 2]),
    index=[f"Actual {name}" for name in CLASS_NAMES],
    columns=[f"Predicted {name}" for name in CLASS_NAMES],
)